# Raster Analysis
### This notebook is designed to help process and analyze the raster data for the change detection analysis.

In [44]:
import os
import arcpy
import numpy as np
from osgeo import gdal
from pathlib import Path

arcpy.env.overwriteOutput = True

### Define Base Paths and Years

In [11]:
# Define paths and parameters
base_raw = Path(r"C:\Users\viver\Documents\git\savannah-port-change-detection\data\raw")
base_out = Path(r"C:\Users\viver\Documents\git\savannah-port-change-detection\data\processed\masked")

years = {
    "2016": "S2A_MSIL2A_2016",
    "2020": "S2A_MSIL2A_2020",
    "2024": "S2B_MSIL2A_2024"
}

nodata_value = -9999

### Masking Logic

In [ ]:
# Process each year
for year, safe_folder in years.items():

    print(f"\n=== Processing {year} ===")

    safe_root = base_raw / safe_folder
    img_data_dirs = list(safe_root.glob("GRANULE/*/IMG_DATA"))

    if not img_data_dirs:
        raise FileNotFoundError(f"No IMG_DATA folder found for {year}")

    img_data = img_data_dirs[0]

    r20m_dir = img_data / "R20m"
    r60m_dir = img_data / "R60m"

    if not r20m_dir.exists():
        raise FileNotFoundError(f"No R20m folder found for {year}")

    # Find any SCL file (20m OR 60m)
    scl_files = list(img_data.rglob("*SCL*.jp2"))

    if not scl_files:
        raise FileNotFoundError(f"No SCL file found for {year}")

    scl_path = scl_files[0]
    print(f"Found SCL: {scl_path}")

    # Choose a reference 20m band for alignment
    ref_band_path = next(p for p in r20m_dir.glob("*.jp2") if "SCL" not in p.name)
    ref_ds = gdal.Open(str(ref_band_path))

    # Resample SCL to 20m if needed
    scl_ds = gdal.Open(str(scl_path))

    if scl_ds.RasterXSize != ref_ds.RasterXSize:
        print("  Resampling SCL to 20m...")

        scl_resampled = gdal.Warp(
            "",
            scl_ds,
            format="MEM",
            width=ref_ds.RasterXSize,
            height=ref_ds.RasterYSize,
            resampleAlg=gdal.GRA_NearestNeighbour,
            outputBounds=ref_ds.GetGeoTransform(),
            dstSRS=ref_ds.GetProjection()
        )
        scl = scl_resampled.GetRasterBand(1).ReadAsArray()
    else:
        scl = scl_ds.GetRasterBand(1).ReadAsArray()

    # Land mask: vegetation + bare soil
    land_mask = np.isin(scl, [4, 5])

    out_dir = base_out / year / "20m"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Mask all 20m bands
    for band_path in r20m_dir.glob("*.jp2"):

        if "SCL" in band_path.name:
            continue

        print(f"  Masking {band_path.name}")

        band_ds = gdal.Open(str(band_path))
        band = band_ds.GetRasterBand(1).ReadAsArray()

        masked = np.where(land_mask, band, nodata_value)

        out_path = out_dir / f"{band_path.stem}_masked.tif"

        driver = gdal.GetDriverByName("GTiff")
        out_ds = driver.Create(
            str(out_path),
            band_ds.RasterXSize,
            band_ds.RasterYSize,
            1,
            gdal.GDT_Float32
        )

        out_ds.SetGeoTransform(band_ds.GetGeoTransform())
        out_ds.SetProjection(band_ds.GetProjection())

        out_band = out_ds.GetRasterBand(1)
        out_band.WriteArray(masked)
        out_band.SetNoDataValue(nodata_value)

        out_band.FlushCache()
        out_ds = None

print("\n✅ All years processed successfully.")

### Clip Rasters to AOI

In [ ]:
# Define paths and parameters
base_in = Path(r"C:\Users\viver\Documents\git\savannah-port-change-detection\data\processed\masked")
base_out = Path(r"C:\Users\viver\Documents\git\savannah-port-change-detection\data\processed\clipped")
aoi = r"C:\Users\viver\OneDrive\Desktop\savannah_port_change_detection\savannah_port_change_detection.gdb\savannah_port_poly"

years = ["2016", "2020", "2024"]

for year in years:
    # Set workspace to current year's masked rasters
    print(f"\n=== Processing {year} ===")
    workspace = os.path.join(base_in, year, "20m")
    arcpy.env.workspace = workspace
    raster_list = arcpy.ListRasters("*")
    
    for raster in raster_list:
        # Clip rastesr to polygon boundary
        arcpy.management.Clip(
        in_raster=raster,
        rectangle="#",
        out_raster= os.path.join(base_out, year, f"{raster[23:-11]}"),
        in_template_dataset=aoi,
        nodata_value="-9999",
        clipping_geometry="ClippingGeometry",
        maintain_clipping_extent="NO_MAINTAIN_EXTENT"
    )
        print(f"  Clipping {raster}")

print("\n✅ All files processed successfully.")

### Convert Bands to GeoTiff

In [ ]:
# Convert clipped rasters to GeoTIFF
base_dir = Path(r"C:\Users\viver\Documents\git\savannah-port-change-detection\data\processed\clipped")
years = ["2016", "2020", "2024"]
bands = ["b04_20m", "b8a_20m"]

for year in years:
    for band in bands:
        in_grid = base_dir / year / band
        out_tif = base_dir / year / f"{band}.tif"

        if out_tif.exists():
            print(f"Skipping {out_tif.name}")
            continue

        print(f"Converting {year}/{band} → GeoTIFF")
        gdal.Translate(
            str(out_tif),
            str(in_grid),
            format="GTiff"
        )

print("✅ Conversion complete")

### Compute NDVI

In [ ]:

# Compute NDVI for each year
base_dir = Path(r"C:\Users\viver\Documents\git\savannah-port-change-detection\data\processed")

years = ["2016", "2020", "2024"]
nodata = -9999

for year in years:
    print(f"Computing NDVI for {year}")

    year_dir = base_dir / "clipped" / year

    red_path = year_dir / "b04_20m.tif"
    nir_path = year_dir / "b8a_20m.tif"

    red_ds = gdal.Open(str(red_path))
    nir_ds = gdal.Open(str(nir_path))

    red = red_ds.GetRasterBand(1).ReadAsArray().astype(float)
    nir = nir_ds.GetRasterBand(1).ReadAsArray().astype(float)

    # Avoid divide-by-zero
    ndvi = np.where(
        (nir + red) == 0,
        nodata,
        (nir - red) / (nir + red)
    )

    out_path = base_dir / "NDVI" / f"NDVI_{year}.tif"

    driver = gdal.GetDriverByName("GTiff")
    out_ds = driver.Create(
        str(out_path),
        red_ds.RasterXSize,
        red_ds.RasterYSize,
        1,
        gdal.GDT_Float32
    )

    out_ds.SetGeoTransform(red_ds.GetGeoTransform())
    out_ds.SetProjection(red_ds.GetProjection())

    out_band = out_ds.GetRasterBand(1)
    out_band.WriteArray(ndvi)
    out_band.SetNoDataValue(nodata)
    out_band.FlushCache()

    out_ds = None

print("✅ NDVI complete for all years.")

Computing NDVI for 2016
Computing NDVI for 2020
Computing NDVI for 2024
✅ NDVI complete for all years.


### Compute ΔNDVI

In [58]:
# Paths
base_dir = Path(r"C:\Users\viver\Documents\git\savannah-port-change-detection\data\processed\NDVI")

ndvi_2016_path = base_dir / "NDVI_2016.tif"
ndvi_2024_path = base_dir / "NDVI_2024.tif"
out_path = base_dir / "NDVI_delta_2016_2024.tif"

nodata = -9999

# Open rasters
ds_2016 = gdal.Open(str(ndvi_2016_path))
ds_2024 = gdal.Open(str(ndvi_2024_path))

ndvi_2016 = ds_2016.GetRasterBand(1).ReadAsArray().astype(float)
ndvi_2024 = ds_2024.GetRasterBand(1).ReadAsArray().astype(float)

# Mask NoData
mask = (
    (ndvi_2016 == nodata) |
    (ndvi_2024 == nodata)
)

# Compute delta NDVI
delta_ndvi = ndvi_2024 - ndvi_2016
delta_ndvi[mask] = nodata

# Write output raster
driver = gdal.GetDriverByName("GTiff")
out_ds = driver.Create(
    str(out_path),
    ds_2016.RasterXSize,
    ds_2016.RasterYSize,
    1,
    gdal.GDT_Float32
)

out_ds.SetGeoTransform(ds_2016.GetGeoTransform())
out_ds.SetProjection(ds_2016.GetProjection())

out_band = out_ds.GetRasterBand(1)
out_band.WriteArray(delta_ndvi)
out_band.SetNoDataValue(nodata)
out_band.FlushCache()

out_ds = None

print("✅ ΔNDVI (2016 → 2024) created successfully")

✅ ΔNDVI (2016 → 2024) created successfully
